# diag0 — BiMamba 역방향 브랜치 상수성 확인

**묻는 것:** BiMamba 의 역방향(backward) 스택이 관측 정보를 실제로 나르는가?

**코드 구조상의 예측:**

- `decoder_in = torch.zeros(...)` 이고 `decoder_pos_embed` 는 `nn.Embedding(K, D)` → 쿼리 `Q` 는 **배치 무관 상수**
- 디코더 입력은 `[C ; Q]` 이고 `C` 만 관측 의존
- 역방향 브랜치는 `flip([C ; Q]) = [q_K…q_1, c_M…c_1]` 을 스캔 → **쿼리가 시퀀스 맨 앞**
- Mamba-2 스캔은 causal (`_causal_conv_step` + `mamba_chunk_scan_combined`)

→ 역방향 스택의 쿼리 위치 출력은 `Q` 만의 함수, 즉 **관측·태스크와 무관한 상수**여야 한다.

| 측정 | 기대 | 뜻 |
|---|---|---|
| `BWD @ query` | `0` | 역방향 출력이 상수 |
| `FWD @ query` | `>> 0` | 민감도 대조군 |
| `출력 action` | `>> 0` | 모델이 살아있음 |

`FWD` 까지 0 이면 두 배치가 사실 같은 관측이라는 뜻이라 **판정 불가**로 빠진다 (거짓 확정 방지).

덤으로 `||bwd|| / ||fwd||` 를 위치별로 잰다. 역방향 기여가 크기 자체로 무시할 만한지 보기 위한 것 —
이 값이 유의미하면 "긴 스캔이 위치 정체성을 잃고 BiMamba 가 head 직전에 복원한다" 가설로 간다.

---

> **커널 얘기.** 이 노트북은 `lerobot` 을 커널로 import 하지 않는다.
> `mamba_ssm` 과 `draccus` 는 실험용 venv 에만 있기 때문에
> (`common_v23.py:43-44`, 기본값 `~/lerobot_project/lerobot_env/bin/python`),
> 계산은 전부 그 venv 로 **subprocess 호출**하고 노트북은 결과 json 만 읽는다.
> 커널은 아무거나 상관없다.


## 0) 부팅 — 실험 venv 찾기

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

_h = Path.cwd()
REPO = next(c for c in (_h, *_h.parents)
            if (c / 'notebooks' / 'libero' / 'diag0_backward_const.py').exists())
sys.path.insert(0, str(REPO / 'notebooks'))

# common_v23 은 stdlib 만 import 한다 → 커널에서 안전하게 불러서 PYTHON 경로만 가져온다.
try:
    import common_v23 as v23
    PYTHON = v23.PYTHON
except Exception as e:
    print('common_v23 import 실패, 기본값 사용:', e)
    PYTHON = os.environ.get('LEROBOT_PYTHON') or str(
        Path.home() / 'lerobot_project' / 'lerobot_env' / 'bin' / 'python')

SCRIPT = REPO / 'notebooks' / 'libero' / 'diag0_backward_const.py'
OUTPUT_BASE = Path(os.environ.get('LEROBOT_OUTPUT',
                                  Path.home() / 'lerobot_project' / 'outputs')) / 'final'
SHARE = OUTPUT_BASE / 'share' / 'diag0'
SHARE.mkdir(parents=True, exist_ok=True)

print('repo   :', REPO)
print('python :', PYTHON, '  (있음)' if Path(PYTHON).exists() else '  ← 없다! 아래 셀 참고')
print('script :', SCRIPT, '  (있음)' if SCRIPT.exists() else '  ← 없다!')
print('share  :', SHARE)

### venv 경로가 틀렸다면

위에서 `python` 이 "없다" 로 나오면 실험 env 를 직접 찾아서 `PYTHON` 을 덮어쓴다.

In [ ]:
# 후보를 훑어본다. 맞는 걸 찾으면 아래 PYTHON = ... 줄에 박아넣으면 된다.
for c in sorted(Path.home().glob('*/*/bin/python')) + sorted(Path.home().glob('*/bin/python')):
    try:
        r = subprocess.run([str(c), '-c', 'import mamba_ssm, draccus; print("ok")'],
                           capture_output=True, text=True, timeout=60)
        print(('OK  ' if r.returncode == 0 else '--  ') + str(c))
    except Exception:
        pass

# PYTHON = '/home/.../lerobot_env/bin/python'   # ← 필요하면 여기서 덮어쓰기

## 1) 설정

`bimamba_pure` 가 논문의 BiMamba 다. **`bimamba` 는 carry 붙은 옛 bimos 이므로 쓰지 않는다.**

In [ ]:
SEED   = 0
STEP   = 150_000      # -1 이면 최신 체크포인트
TASK   = 'libero_10'
BATCH  = 4            # 관측이 서로 다른 샘플 수
GPU    = '0'          # CUDA_VISIBLE_DEVICES
STAMP  = time.strftime('%Y%m%d_%H%M')

ENV = dict(os.environ,
           PYTHONPATH=str(REPO / 'src'),
           HF_HUB_DISABLE_XET='1',
           MPLBACKEND='Agg',
           CUDA_VISIBLE_DEVICES=GPU)

def run_diag(tags, json_path):
    """실험 venv 로 diag0 을 돌리고 출력을 실시간으로 흘린다."""
    cmd = [PYTHON, str(SCRIPT), '--tags', tags, '--seed', str(SEED), '--step', str(STEP),
           '--task', TASK, '--batch', str(BATCH), '--json', str(json_path)]
    print('$', ' '.join(cmd), '\n')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=ENV, cwd=str(REPO))
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    print(f'\n[exit {rc}]')
    return json.loads(Path(json_path).read_text(encoding='utf-8')) if Path(json_path).exists() else {}

## 2) K=100 단일 확인 — 이 셀 하나가 0단계의 답이다

In [ ]:
res100 = run_diag('bimamba_pure', SHARE / f'diag0_k100_{STAMP}.json')

## 3) K 별로 — 50 / 100 / 150

체크포인트가 없는 K 는 건너뛴다. 역방향 기여 비율이 K 에 따라 커지면,
그 자체로 성공률 이득의 K 의존성(+2.2 → +8.2 → +9.0)을 설명하는 근거가 된다.

한 프로세스 안에서 도니까 데이터셋을 K 별로 한 번만 읽는다.

In [ ]:
results = run_diag('all', SHARE / f'diag0_all_{STAMP}.json')
print('\n받은 태그:', list(results))

## 4) 요약표

In [ ]:
import csv

K_OF = {'bimamba_pure_k50': 50, 'bimamba_pure': 100, 'bimamba_pure_k150': 150}
rows = []
hdr = f"{'K':>5} {'tag':<20} {'판정':<14} {'BWD A-B':>11} {'FWD A-B':>11} {'ratio 평균':>11}"
print(hdr); print('-' * len(hdr))
for tag, r in sorted(results.items(), key=lambda kv: K_OF.get(kv[0], 0)):
    rat = sum(r['ratio']) / len(r['ratio']) if r.get('ratio') else float('nan')
    print(f"{r.get('K', K_OF.get(tag, 0)):>5} {tag:<20} {r['verdict']:<14} "
          f"{r.get('bwd_ab', float('nan')):11.3e} {r.get('fwd_ab', float('nan')):11.3e} {rat:11.3f}")
    rows.append({'K': r.get('K', K_OF.get(tag)), 'tag': tag, 'verdict': r['verdict'],
                 'bwd_ab': r.get('bwd_ab'), 'bwd_spread': r.get('bwd_spread'),
                 'fwd_ab': r.get('fwd_ab'), 'fwd_spread': r.get('fwd_spread'),
                 'act_ab': r.get('act_ab'), 'ratio_mean': rat})

if rows:
    csv_path = SHARE / f'diag0_summary_{STAMP}.csv'
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    print('\nsaved', csv_path)

## 5) 위치별 역방향 기여 — `||bwd|| / ||fwd||`

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for tag, r in sorted(results.items(), key=lambda kv: K_OF.get(kv[0], 0)):
    y = r.get('ratio')
    if not y:
        continue
    ax.plot(range(1, len(y) + 1), y, label=f"K={r.get('K', K_OF.get(tag))}", lw=1.6)
ax.set_xlabel('청크 내 위치 k'); ax.set_ylabel('||bwd|| / ||fwd||')
ax.set_title('역방향 브랜치 기여 크기 (위치별)')
ax.grid(alpha=.3); ax.legend()
fig.tight_layout()
png = SHARE / f'diag0_ratio_{STAMP}.png'
fig.savefig(png, dpi=150, bbox_inches='tight')
print('saved', png)
plt.show()

## 6) 결과 읽는 법

**`BWD A-B == 0` 이고 `배치 내 샘플 간 == 0`** → 확정. 역방향 스택은 정보를 나르지 않고
head 직전에 위치별 상수벡터 `b_k` 를 더하는 역할만 한다. 원고의 "앞쪽 쿼리가 뒤쪽을 본다"
설명은 이 구현에서 성립하지 않으며, 1단계 위치별 오차 분석의 해석 틀이 여기서 정해진다.

**`ratio` 가 0.01 수준** → 역방향 기여 자체가 미미하다는 뜻. 위치 정체성 가설이 약해지고,
9.0 포인트 차이의 원인을 다른 데서 찾아야 한다.

**`ratio` 가 0.3~1 수준이고 K 에 따라 커짐** → 가설이 강해진다. 다음은 2단계 ablation —
역방향 스택을 `nn.Embedding(K, 512)` 하나로 대체해서 이득이 회복되는지 본다.

**`반증`** → 구조 분석이 틀렸다. 스캔이 causal 이 아니거나 쿼리가 관측 의존이라는 뜻이므로
`modeling_acm2_sscp_literal.py` 의 `mamba2_stateful_forward` 경로를 다시 봐야 한다.

산출물은 `outputs/final/share/diag0/` 에 json · csv · png 로 남는다.